In [ ]:
# Komórka 1: Konfiguracja Środowiska i Instalacja Zależności

import os

# Krok 1: Klonowanie repozytorium z modułami
print(" cloning repository...")
!git clone https://github.com/MattyMroz/ColabUniversalDownloader.git

# Przejście do katalogu repozytorium
os.chdir('ColabUniversalDownloader')
print(f" Zmieniono katalog roboczy na: {os.getcwd()}")

# Krok 2: Instalacja zależności z pliku requirements.txt
print("\n🐍 Instalowanie bibliotek Pythona z requirements.txt...")
!pip install -q -r requirements.txt
print("✅ Biblioteki Pythona zainstalowane pomyślnie.")

print("\n\n🎉 Środowisko jest gotowe do testów!")

In [ ]:
#@title Ustawienia (Form)

# Ścieżki katalogów tymczasowych
TMP_PIXELDRAIN = "./tmp_pixeldrain"  #@param {type:"string"}
TMP_MEGA = "./tmp_mega"  #@param {type:"string"}

# Parametry testów
DELETE_DELAY_SECONDS = 15  #@param {type:"number"}
CLEANUP_TMP_AFTER_TESTS = True  #@param {type:"boolean"}

# Linki testowe (możesz zmienić)
PIXEL_URL = "https://pixeldrain.com/u/e75isJ7y"  #@param {type:"string"}
MEGA_FILE_URL = "https://mega.nz/file/CVQXjbCZ#KhWlLh7Ec3z0EjcPqVX1_3ZyGQC05xNMs8gb_VYGdxg"  #@param {type:"string"}
MEGA_FOLDER_URL = "https://mega.nz/folder/SVxnDbbI#V5nwTs9FAXNlGzUmWBucIw"  #@param {type:"string"}


In [ ]:
# Komórka 2: Import Modułów i Inicjalizacja Narzędzi

from utils.google_drive import GoogleDriveManager
from utils.mega import MegaDownloader
from utils.pixeldrain import PixelDrainDownloader
import sys
from pprint import pprint

# Dodajemy ścieżkę do repozytorium, aby Python widział nasze moduły w katalogu 'utils'
sys.path.append('/content/ColabUniversalDownloader')


# --- Inicjalizacja klas ---
pixeldrain_handler = PixelDrainDownloader()
mega_handler = MegaDownloader()
gdrive_handler = GoogleDriveManager()

print("✅ Moduły zaimportowane, obiekty gotowe do pracy.")
print(f"Google Drive Manager gotowy: {gdrive_handler.is_ready()}")

# --- Prosta funkcja do wyświetlania postępu ---


def simple_progress_callback(log_message):
    """Wyświetla każdą nową linię logu."""
    print(log_message)

In [ ]:
#@title Helpery (unikalne nazwy, filtrowanie logów, bez nadpisywania)
import os, re, time, shutil, uuid
from pathlib import Path

# Unikalny znacznik sesji, aby nie nadpisywać
SESSION_ID = time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]


def ensure_unique_dir(base_dir: str, label: str) -> str:
    """Zwraca ścieżkę do unikalnego podkatalogu w base_dir: base_dir/label-<SESSION_ID>."""
    p = Path(base_dir) / f"{label}-{SESSION_ID}"
    p.mkdir(parents=True, exist_ok=True)
    return str(p)


def safe_join(base_dir: str, filename: str) -> str:
    """Łączy katalog i nazwę pliku, nie pozwalając na wyjście ponad base_dir."""
    base = Path(base_dir).resolve()
    full = (base / filename).resolve()
    if base not in full.parents and base != full:
        # Jeśli ścieżka wychodzi poza katalog bazowy, przerzuć do bazowego
        full = base / Path(filename).name
    return str(full)


# Mega: filtr komunikatów do najważniejszych (postęp + podsumowania)
IMPORTANT_PATTERNS = [
    r"^\s*\d+%",             # linie postępu
    r"Downloaded ",           # ukończenia pobrań
    r"Downloading ",          # start pobierania
    r"^Error|^BŁĄD|^❌|^⚠️",  # błędy/ostrzeżenia
]
IMPORTANT_REGEX = re.compile("|".join(IMPORTANT_PATTERNS), re.IGNORECASE)

def mega_progress_minimal(line: str):
    if IMPORTANT_REGEX.search(line):
        print(line, end="")


# Pixeldrain: zbieraj do bufora i drukuj raz po zakończeniu
class SingleLineLogger:
    def __init__(self):
        self._buf = []
    def __call__(self, line: str):
        # ogranicz objętość bufora
        if len(self._buf) < 2000:
            self._buf.append(line.strip())
    def flush(self, header: str = None):
        if header:
            print(header)
        if self._buf:
            # pokaż tylko ostatnią linię lub streszczenie
            last = self._buf[-1]
            print(last)
        self._buf.clear()

single_line_logger = SingleLineLogger()


In [ ]:
#@title Test 1: Pixeldrain
# Komórka 3: Test Modułu `pixeldrain.py` — z użyciem katalogu tymczasowego

import os
import shutil

print("="*50)
print("🚀 TEST 1: POBIERANIE Z PIXELDRAIN 🚀")
print("="*50)

# Katalog tymczasowy (unikalny podkatalog sesji)
PXD_TMP_BASE = TMP_PIXELDRAIN
PXD_TMP = ensure_unique_dir(PXD_TMP_BASE, "pixeldrain")
os.makedirs(PXD_TMP, exist_ok=True)

filepath = None
prev_cwd = os.getcwd()

try:
    # Pobieraj do katalogu tymczasowego (wget zapisze w aktualnym katalogu procesu)
    os.chdir(PXD_TMP)
    # Zbierz log i pokaż tylko jedną linię podsumowania
    filepath = pixeldrain_handler.download(PIXEL_URL, single_line_logger)

    if filepath and os.path.exists(filepath):
        single_line_logger.flush(header="\n✅ ZAKOŃCZONO POBIERANIE PIXELDRAIN")
        print(f"Plik: {filepath}")
        try:
            print(f"Rozmiar: {os.path.getsize(filepath) / 1024:.2f} KB")
        except Exception:
            pass
    else:
        single_line_logger.flush(header="\n❌ POBIERANIE PIXELDRAIN NIEUDANE")

finally:
    # Przywróć katalog roboczy i posprzątaj katalog tymczasowy (jeśli włączone)
    try:
        os.chdir(prev_cwd)
    except Exception:
        pass
    if CLEANUP_TMP_AFTER_TESTS:
        shutil.rmtree(PXD_TMP, ignore_errors=True)


In [ ]:
#@title Test 2: Mega.nz
# Komórka 4: Test Modułu `mega.py` — pobieranie pliku i całego folderu (bez listowania)

import os

print("="*50)
print("🚀 TEST 2: MEGA.NZ — PLIK I FOLDER 🚀")
print("="*50)

# Jeden wspólny katalog tymczasowy (unikalny podkatalog sesji)
MEGA_TMP_BASE = TMP_MEGA
MEGA_TMP = ensure_unique_dir(MEGA_TMP_BASE, "mega")
os.makedirs(MEGA_TMP, exist_ok=True)

# --- Test A: Pobieranie pojedynczego pliku ---
print("\n--- TEST 2A: Pobieranie pliku ---")
file_path = mega_handler.download_file(
    MEGA_FILE_URL, dest_dir=MEGA_TMP, progress=mega_progress_minimal)
if file_path and os.path.exists(file_path):
    print(f"\n✅ PLIK POBRANY: {file_path}")
    try:
        size_mb = os.path.getsize(file_path) / (1024*1024)
        print(f"   Rozmiar: {size_mb:.2f} MB")
    except Exception:
        pass
else:
    print("\n❌ Nie udało się pobrać pliku (sprawdź URL).")

# --- Test B: Pobieranie całego folderu ---
print("\n--- TEST 2B: Pobieranie folderu (całość) ---")
results = mega_handler.download_folder(
    MEGA_FOLDER_URL, dest_dir=MEGA_TMP, choose_files=False, progress=mega_progress_minimal)

if results:
    print("\n✅ Pobrane pliki (wykryte w logu):")
    for p in results:
        print(f" - {p}")
else:
    # Fallback: wypisz cokolwiek pobrało się do folderu docelowego
    collected = []
    for root, _, files in os.walk(MEGA_TMP):
        for f in files:
            collected.append(os.path.join(root, f))
    if collected:
        print("\nℹ️ Nie wykryto plików w logu, ale znaleziono w systemie:")
        for p in collected:
            print(f" - {p}")
    else:
        print("\n❌ Nie udało się pobrać folderu (sprawdź URL).")

print("\n🎯 Zakończono testy Mega.")


In [ ]:
# Komórka 5: Test Modułu `google_drive.py`

print("="*50)
print("🚀 TEST 3: OPERACJE NA GOOGLE DRIVE 🚀")
print("="*50)
print("UWAGA: Ta komórka wywoła okno autoryzacji Google. Zezwól na dostęp.")

if not gdrive_handler.is_ready():
    print("\n❌ TEST NIEUDANY. Manager Dysku Google nie został poprawnie zainicjalizowany.")
else:
    DUMMY_FILENAME = "test_upload.txt"
    DUMMY_CONTENT = "To jest plik testowy z Colab Universal Downloader."
    file_id_to_delete = None

    try:
        # Tworzenie pliku testowego
        with open(DUMMY_FILENAME, "w") as f:
            f.write(DUMMY_CONTENT)
        print(f"Stworzono plik testowy: {DUMMY_FILENAME}")

        # Krok 1: Pobierz ID "Mojego Dysku"
        print("\n--> Testowanie get_drive_id()...")
        parent_id = gdrive_handler.get_drive_id(
            drive_name="Mój Dysk", is_shared=False)
        assert parent_id == 'root'
        print("✅ ID 'Mojego Dysku' pobrane poprawnie ('root').")

        # Krok 2: Wyślij plik i udostępnij
        print("\n--> Testowanie upload_and_share()...")
        upload_info = gdrive_handler.upload_and_share(
            DUMMY_FILENAME, parent_id)

        if upload_info and upload_info.get('link'):
            print(f"✅ Plik wysłany i udostępniony pomyślnie!")
            print(f"   Link do pobrania: {upload_info['link']}")
            file_id_to_delete = upload_info['id']
        else:
            raise AssertionError(
                "Nie udało się wysłać pliku lub uzyskać linku.")

        # Krok 3: Zaplanuj usunięcie pliku (z parametru)
        if file_id_to_delete:
            print("\n--> Testowanie delete_file_after_delay()...")
            gdrive_handler.delete_file_after_delay(
                file_id_to_delete, delay_seconds=int(DELETE_DELAY_SECONDS))
            print(f"✅ Zadanie usunięcia pliku zaplanowane na za {DELETE_DELAY_SECONDS} sekund.")
            print("   Możesz sprawdzić na Dysku Google, czy plik zniknie.")

    except Exception as e:
        print(f"\n❌ WYSTĄPIŁ BŁĄD PODCZAS TESTU: {e}")
    finally:
        # Sprzątanie lokalnego pliku
        if os.path.exists(DUMMY_FILENAME):
            os.remove(DUMMY_FILENAME)
            print(f"\n🧹 Usunięto lokalny plik testowy: {DUMMY_FILENAME}")
